<!-- SPDX-License-Identifier: CC-BY-4.0 -->
<!-- Copyright Contributors to the ODPi Egeria project 2024. -->

![Egeria Logo](https://raw.githubusercontent.com/odpi/egeria/main/assets/img/ODPi_Egeria_Logo_color.png)

### Coco Pharmaceuticals Lab
----

# Adding the employees from the acquired sites

## Introduction

Coco Pharmaceuticals has grown through two acquisitions:

* **Austin Pharmaceuticals, Inc.** - now the Austin Site in Texas.
* **EKG Pharmaceuticals S.R.L.** - now the Bucharest Site in Romania.  This acquisition has only just closed.

Both companies are principally manufacturers, and both specialize in manufacturing personalized medicines - small batches made to flexible recipes, often for an individual patient.  Austin makes cell therapies and named-patient products.  Bucharest makes sterile products and autologous serum eye drops, which are prepared from the patient's own blood.

Each company arrived with its own systems, and its own people.  The systems have been catalogued in the IT Systems Inventory (see [extending-the-systems-inventory.ipynb](../../1.%20coco-data-hub/extending-the-systems-inventory/extending-the-systems-inventory.ipynb)), but their business owners are recorded there only as names.  Coco's governance programs need to reach these people - as system owners, as members of the sustainability community and, for procurement, as part of the effort to consolidate the supplier lists across the company.  So they need profiles in Egeria.

This notebook builds them out, following the pattern used for the existing Coco employees:

1. [Tom Tally](https://egeria-project.org/practices/coco-pharmaceuticals/personas/tom-tally/) extends the reference data sets so that people and locations can be described as being in Bucharest, Romania.  This is the *Add Bucharest location definitions* task from the sustainability project.
2. [Faith Broker](https://egeria-project.org/practices/coco-pharmaceuticals/personas/faith-broker/) from Human Resources creates the teams for the Bucharest Site.
3. Faith adds each employee with their actor profile, user identity, contact details and team role.
4. [Gary Geeke](https://egeria-project.org/practices/coco-pharmaceuticals/personas/gary-geeke/), who looks after the IT Systems Inventory, appoints the system owners.

[adding-procurement.ipynb](adding-procurement.ipynb) runs after this notebook.

----

## Setting up

The first code cell sets up the connection to Egeria.  If you are using the standard quickstart environment then there is no need to change these values.

----

In [ ]:
# Initialize pyegeria

import os
view_server = os.environ.get("EGERIA_VIEW_SERVER","qs-view-server")
url = os.environ.get("EGERIA_VIEW_SERVER_URL","https://host.docker.internal:9443")
user_pwd = os.environ.get("EGERIA_USER_PASSWORD")

toms_user_id = "tomtally"
faiths_user_id = "faithbroker"
garys_user_id = "garygeeke"

print("\n")
print("Accessing view server " + view_server + " on platform " + url + " for users " + toms_user_id + ", " + faiths_user_id + " and " + garys_user_id)
print("\n")

from IPython.display import display, Markdown
import csv
import json
import unicodedata

from pyegeria import (ReferenceDataManager, ValidMetadataManager, ClassificationExplorer,
                      ActorManager, PeopleOrganizer, CollectionManager)
from pyegeria.core._globals import NO_ELEMENTS_FOUND

# Tom maintains the reference data used to describe people and locations
toms_ref_data_client = ReferenceDataManager(view_server, url, toms_user_id, user_pwd)
toms_ref_data_client.create_egeria_bearer_token()

toms_valid_metadata_client = ValidMetadataManager(view_server, url, toms_user_id, user_pwd)
toms_valid_metadata_client.create_egeria_bearer_token()

toms_explorer_client = ClassificationExplorer(view_server, url, toms_user_id, user_pwd)
toms_explorer_client.create_egeria_bearer_token()

# Faith maintains the people and teams
faiths_actor_client = ActorManager(view_server, url, faiths_user_id, user_pwd)
faiths_actor_client.create_egeria_bearer_token()

faiths_people_client = PeopleOrganizer(view_server, url, faiths_user_id, user_pwd)
faiths_people_client.create_egeria_bearer_token()

faiths_collection_client = CollectionManager(view_server, url, faiths_user_id, user_pwd)
faiths_collection_client.create_egeria_bearer_token()

# Gary appoints the owners of the systems in the IT Systems Inventory
garys_actor_client = ActorManager(view_server, url, garys_user_id, user_pwd)
garys_actor_client.create_egeria_bearer_token()

print("Connected to Egeria")

----

Everything this notebook creates is located through its unique name (qualifiedName), so that the notebook can be rerun without creating duplicates.

----

In [ ]:
def get_guid(qualified_name: str) -> str | None:
    guid = toms_explorer_client.get_element_guid_by_unique_name(qualified_name)
    if guid is None or guid == NO_ELEMENTS_FOUND:
        return None
    return guid


def related_qualified_names(element: dict, relationship_list_name: str) -> list[str]:
    return [related["relatedElement"]["properties"].get("qualifiedName")
            for related in (element.get(relationship_list_name) or [])]

----

## A guide: Gary Geeke's profile

The existing Coco employees were loaded from the Coco Pharmaceuticals open metadata archive.  Gary Geeke's profile shows the pattern to follow.

* The *Person* entity holds the name, job title, employee number and type of contract.  Its `additionalProperties` record the person's work location, country, organization and job level.  The values of these properties come from reference data sets.  For example, `workLocation` holds the preferred value of a value from the *WorkLocation* reference data set - `1` is Amsterdam.
* A *UserIdentity* links Gary's user id to the profile.
* A *ContactDetails* entity holds Gary's work email address.
* Gary's membership of teams is shown through *PersonRoleAppointment* relationships to the team's leader and member roles.

----

In [ ]:
garys_profile = garys_actor_client.get_actor_profile_by_guid(get_guid("Person::NL::199995"), output_format="JSON")

garys_properties = garys_profile["properties"]
print(f"{garys_properties['displayName']} - {garys_properties['jobTitle']}")
print(f"additionalProperties: {garys_properties['additionalProperties']}")
print()
for contact in garys_profile.get("contactDetails", []):
    contact_properties = contact["relatedElement"]["properties"]
    print(f"{contact_properties['displayName']}: {contact_properties['contactMethodValue']}")
print()
for role in garys_profile.get("performsRoles", [])[:6]:
    print("Performs role: " + role["relatedElement"]["properties"]["qualifiedName"])

----

## Step 1: Extending the reference data sets

The Austin Site is already described in the reference data, but Bucharest is not.  Tom adds:

* **Romania** to the *CountryCode* reference data set, with the preferred value `RO`.
* **Bucharest** to the *WorkLocation* reference data set, with the preferred value `12` and the postal address of the site as its description.

Each value is held twice: as a *ReferenceDataValue* that is a member of the reference data set, and as a *ValidMetadataValue* for the matching map name in `additionalProperties`.  The valid metadata value is the one Egeria uses when validating the property values.

----

In [ ]:
def add_reference_data_value(set_qualified_name: str, value_name: str, preferred_value: str, description: str | None) -> str:
    set_guid = get_guid(set_qualified_name)
    reference_data_set = toms_explorer_client.get_element_by_guid(set_guid)
    set_properties = reference_data_set["properties"]

    value_qualified_name = "ReferenceDataSet::" + value_name
    value_guid = get_guid(value_qualified_name)
    if value_guid:
        print(f"{value_qualified_name} already exists")
        return value_guid

    ordinal = len(reference_data_set.get("validValueMembers") or [])
    body = {
        "class": "NewElementRequestBody",
        "anchorGUID": set_guid,
        "isOwnAnchor": False,
        "properties": {
            "class": "ValidValueDefinitionProperties",
            "typeName": "ReferenceDataValue",
            "qualifiedName": value_qualified_name,
            "identifier": set_properties.get("identifier"),
            "displayName": value_name,
            "description": description,
            "category": set_properties.get("category"),
            "usage": set_properties.get("usage"),
            "scope": set_properties.get("scope"),
            "preferredValue": preferred_value,
            "dataType": "string",
            "isCaseSensitive": False,
        },
    }
    value_guid = toms_ref_data_client.create_valid_value_definition(body)

    member_body = {
        "class": "NewRelationshipRequestBody",
        "properties": {
            "class": "ValidValueMemberProperties",
            "isDefaultValue": False,
            "ordinal": ordinal,
        },
    }
    toms_ref_data_client.link_valid_value_definition(set_guid, value_guid, member_body)
    print(f"{value_qualified_name} added to {set_qualified_name} with preferred value {preferred_value}")
    return value_guid


def add_valid_metadata_value(map_name: str, display_name: str, preferred_value: str, description: str | None) -> str:
    body = {
        "displayName": display_name,
        "description": description,
        "preferredValue": preferred_value,
        "dataType": "string",
        "isCaseSensitive": True,
    }
    toms_valid_metadata_client.setup_valid_metadata_map_value("additionalProperties", map_name, None, body)
    valid_metadata_value_qualified_name = f"Egeria:ValidMetadataValue::additionalProperties--{map_name}--({preferred_value})"
    print(f"{valid_metadata_value_qualified_name} set up")
    return get_guid(valid_metadata_value_qualified_name)

In [ ]:
romania_ref_data_guid = add_reference_data_value("ReferenceDataSet::CountryCode", "Romania", "RO", None)
romania_valid_metadata_guid = add_valid_metadata_value("countryCode", "Romania", "RO", None)

----

The work location values are linked to the site location they describe through the *MoreInformation* relationship.  The Bucharest Site was created when its systems were added to the IT Systems Inventory.

----

In [ ]:
bucharest_site_qualified_name = "Site::BUC-SITE"
bucharest_site_guid = get_guid(bucharest_site_qualified_name)
bucharest_site = toms_explorer_client.get_element_by_guid(bucharest_site_guid)
bucharest_postal_address = None
for location_kind in bucharest_site["elementHeader"].get("locationKinds", []):
    if location_kind["classificationName"] == "FixedLocation":
        bucharest_postal_address = location_kind["classificationProperties"].get("postalAddress")
print(f"{bucharest_site_qualified_name}: {bucharest_postal_address}")

bucharest_ref_data_guid = add_reference_data_value("ReferenceDataSet::WorkLocation", "Bucharest", "12", bucharest_postal_address)
bucharest_valid_metadata_guid = add_valid_metadata_value("workLocation", "Bucharest", "12", bucharest_postal_address)

for value_guid in [bucharest_ref_data_guid, bucharest_valid_metadata_guid]:
    value = toms_explorer_client.get_element_by_guid(value_guid)
    if bucharest_site_qualified_name in related_qualified_names(value, "providesMoreInformation"):
        print(f"{value['properties']['qualifiedName']} is already linked to {bucharest_site_qualified_name}")
    else:
        # pyegeria 6.2.10 sends the wrong properties class by default, so the body is passed explicitly
        more_information_body = {
            "class": "NewRelationshipRequestBody",
            "properties": {"class": "MoreInformationProperties"},
        }
        toms_explorer_client.add_more_information(bucharest_site_guid, value_guid, more_information_body)
        print(f"{value['properties']['qualifiedName']} linked to {bucharest_site_qualified_name}")

----

## Step 2: Creating the Bucharest teams

The Austin Site already has teams: the *Austin Site* team for all Austin employees, with the *Austin Factory*, *Austin Office Team* and *Austin Information Technology Team* departments within it.  Faith creates the same structure for Bucharest from [data/bucharest_teams.csv](data/bucharest_teams.csv).

Each team follows the pattern of the existing departments:

* The team is a *Team* profile with the unique name `Department::<team id>`.
* It sits within its parent team through the *TeamStructure* relationship.  The *Bucharest Site* team sits within the *Founders* team, like the other site teams.
* It has a team leader role and a team member role.  These are *PersonRole* elements linked to the team with the *AssignmentScope* relationship.  People join the team by being appointed to one of these roles.
* Departments are owned by a business area.  The business capability that represents the business area has the team as a member.

----

In [ ]:
with open("data/bucharest_teams.csv", newline="", encoding="utf-8") as csv_file:
    bucharest_teams = list(csv.DictReader(csv_file))


def create_team_role(team_guid: str, role_qualified_name: str, role_group: str, assignment_type: str, head_count: int):
    role_guid = get_guid(role_qualified_name)
    if role_guid:
        return role_guid

    body = {
        "class": "NewElementRequestBody",
        "isOwnAnchor": True,
        "properties": {
            "class": "PersonRoleProperties",
            "typeName": "PersonRole",
            "qualifiedName": role_qualified_name,
            "identifier": role_qualified_name,
            "actorRoleGroups": [role_group],
            "headCountLimitSet": True,
            "headCount": head_count,
        },
    }
    role_guid = faiths_actor_client.create_actor_role(body)
    scope_body = {
        "class": "NewRelationshipRequestBody",
        "properties": {"class": "AssignmentScopeProperties", "assignmentType": assignment_type},
    }
    faiths_actor_client.link_assignment_scope(team_guid, role_guid, scope_body)
    return role_guid


for team in bucharest_teams:
    team_qualified_name = "Department::" + team["team_id"]
    team_guid = get_guid(team_qualified_name)
    if team_guid:
        print(f"{team_qualified_name} {team['display_name']} already exists")
        continue

    body = {
        "class": "NewElementRequestBody",
        "isOwnAnchor": True,
        "properties": {
            "class": "TeamProperties",
            "typeName": "Team",
            "qualifiedName": team_qualified_name,
            "identifier": team["team_id"],
            "displayName": team["display_name"],
            "description": team["description"],
            "teamType": "Dept",
        },
    }
    team_guid = faiths_actor_client.create_actor_profile(body)

    structure_body = {
        "class": "NewRelationshipRequestBody",
        "properties": {"class": "TeamStructureProperties", "delegationEscalationAuthority": True},
    }
    faiths_people_client.link_team_structure(get_guid("Department::" + team["super_team_id"]), team_guid, structure_body)

    create_team_role(team_guid, "TeamLeader::" + team_qualified_name, "team-leader", "Leader", 1)
    create_team_role(team_guid, "TeamMembers::" + team_qualified_name, "team-member", "Contributor", 0)

    if team["business_area"]:
        membership_body = {
            "class": "NewRelationshipRequestBody",
            "properties": {
                "class": "CollectionMembershipProperties",
                "membershipType": "Owner",
                "membershipRationale": team["business_area_scope"],
            },
        }
        faiths_collection_client.add_to_collection(get_guid("BusinessArea::" + team["business_area"]), team_guid, membership_body)

    print(f"{team_qualified_name} {team['display_name']} created")

----

## Step 3: Adding the employees

The employees come from three files in the [data](data) folder:

* [austin_employees.csv](data/austin_employees.csv) - the Austin staff.  Most of them come from Austin's Workday HCM system.  The rest are the business owners named in the Austin system inventory, who are not in the Workday extract.
* [ekg_romanian_employees.csv](data/ekg_romanian_employees.csv) - the EKG staff list provided at the acquisition.  It lists the office, technical and leadership staff.
* [ekg_additional_employees.csv](data/ekg_additional_employees.csv) - the factory and laboratory staff from EKG's Workday HCM system who are not on the staff list, plus Carmen Iliescu, who looks after procurement at Bucharest.  Several of the shop-floor staff have no work email address.

The EKG files do not say which team each person belongs to, so [ekg_employee_assignments.csv](data/ekg_employee_assignments.csv) supplies the team, team role, job level and employee type for each EKG employee.

Two choices follow the existing Coco profiles:

* Each person's unique name is `Person::<country code>::<employee number>`.
* Each person's user id is their first name and surname, in lower case, without accents.

Where a file does not give a person's pronouns or courtesy title, they are left empty rather than guessed.  The date of birth in the EKG staff list is not needed in the catalog, so it is not loaded.

----

In [ ]:
def read_csv(file_name: str) -> list[dict]:
    with open("data/" + file_name, newline="", encoding="utf-8") as csv_file:
        return list(csv.DictReader(csv_file))


employees = []

for row in read_csv("austin_employees.csv"):
    row.update({"country_code": "USA", "work_location": "7",
                "email_service": "Email address assigned by Austin Pharmaceuticals, Inc."})
    employees.append(row)

ekg_assignments = {row["employee_number"]: row for row in read_csv("ekg_employee_assignments.csv")}
for row in read_csv("ekg_romanian_employees.csv") + read_csv("ekg_additional_employees.csv"):
    row.update(ekg_assignments[row["employee_number"]])
    row.update({"country_code": "RO", "work_location": "12",
                "email_service": "Email address assigned by EKG Pharmaceuticals S.R.L."})
    employees.append(row)

print(f"{len(employees)} employees to add")

----

The function below creates the profile for one employee, together with the user identity and the contact details that hang off it.  Both are anchored to the profile, so they are created in the same request as their link to it, using the `parentGUID` fields of the request.  Each of the three elements is checked for separately, so a rerun completes an employee whose details were only partly created.

----

In [ ]:
def ascii_name(name: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", name) if c.isascii() and c.isalpha()).lower()


def add_employee(employee: dict) -> str:
    person_qualified_name = f"Person::{employee['country_code']}::{employee['employee_number']}"
    given_name = employee["first_name"]
    initials = " ".join([given_name[0]] + [i.strip(".") for i in employee["middle_initials"].split() if i])
    full_name = " ".join(part for part in [employee["courtesy_title"], given_name, employee["middle_initials"], employee["surname"]] if part)

    profile_body = {
        "class": "NewElementRequestBody",
        "isOwnAnchor": True,
        "properties": {
            "class": "PersonProperties",
            "typeName": "Person",
            "qualifiedName": person_qualified_name,
            "displayName": given_name + " " + employee["surname"],
            "courtesyTitle": employee["courtesy_title"] or None,
            "initials": initials,
            "givenNames": given_name,
            "surname": employee["surname"],
            "fullName": full_name,
            "pronouns": employee["pronouns"] or None,
            "jobTitle": employee["job_title"],
            "employeeNumber": employee["employee_number"],
            "employeeType": employee["employee_type"],
            "additionalProperties": {
                "workLocation": employee["work_location"],
                "countryCode": employee["country_code"],
                "organization": "Coco Pharmaceuticals Ltd",
                "jobLevel": employee["job_level"],
            },
        },
    }
    person_guid = get_guid(person_qualified_name)
    if person_guid is None:
        person_guid = faiths_actor_client.create_actor_profile(profile_body)

    user_id = ascii_name(given_name) + ascii_name(employee["surname"])
    distinguished_name = f"cn={user_id},ou=people,ou=users,o=cocoPharma"
    identity_body = {
        "class": "NewElementRequestBody",
        "anchorGUID": person_guid,
        "isOwnAnchor": False,
        "parentGUID": person_guid,
        "parentRelationshipTypeName": "ProfileIdentity",
        "parentAtEnd1": True,
        "properties": {
            "class": "UserIdentityProperties",
            "typeName": "UserIdentity",
            "qualifiedName": "UserIdentity:" + distinguished_name,
            "userId": user_id,
            "distinguishedName": distinguished_name,
        },
    }
    if get_guid(identity_body["properties"]["qualifiedName"]) is None:
        faiths_actor_client.create_user_identity(identity_body)

    contact_qualified_name = "ContactDetails::" + person_qualified_name + "::Work Email"
    if employee["email_address"] and get_guid(contact_qualified_name) is None:
        contact_body = {
            "class": "NewElementRequestBody",
            "anchorGUID": person_guid,
            "isOwnAnchor": False,
            "parentGUID": person_guid,
            "parentRelationshipTypeName": "ContactThrough",
            "parentAtEnd1": True,
            "properties": {
                "class": "ContactDetailsProperties",
                "typeName": "ContactDetails",
                "qualifiedName": contact_qualified_name,
                "displayName": "Work Email Address",
                "contactType": "Work Email",
                "contactMethodType": "EMAIL",
                "contactMethodService": employee["email_service"],
                "contactMethodValue": employee["email_address"],
            },
        }
        faiths_actor_client.create_contact_details(contact_body)

    return person_guid

In [ ]:
person_guids = {}
for employee in employees:
    already_exists = get_guid(f"Person::{employee['country_code']}::{employee['employee_number']}") is not None
    person_guids[employee["employee_number"]] = add_employee(employee)
    print(f"{employee['employee_number']:8} {employee['first_name']} {employee['surname']} - {employee['job_title']}" + (" (already exists)" if already_exists else ""))

print(f"\n{len(person_guids)} employee profiles are in Egeria")

----

Now each employee is appointed to the leader or member role of their team.  Former employees are not appointed to any team.

Each team role has a head count.  The places in the Austin member roles are already taken by the anonymous employees that the archive created to represent the rest of the staff.  Egeria treats the head count as information for the people managing the team rather than a limit, so the named employees can still be appointed.  The Austin roles belong to the archive, so their head counts are left alone.  The head counts of the Bucharest roles, which this notebook created, are set to the number of people appointed.

----

In [ ]:
bucharest_team_ids = [team["team_id"] for team in bucharest_teams]

appointments = {}
for employee in employees:
    if employee["team_id"]:
        role_prefix = "TeamLeader::" if employee["team_role"] == "leader" else "TeamMembers::"
        appointments.setdefault(role_prefix + "Department::" + employee["team_id"], []).append(employee)

for role_qualified_name, appointees in appointments.items():
    role_guid = get_guid(role_qualified_name)
    role = faiths_actor_client.get_actor_role_by_guid(role_guid, output_format="JSON")
    current_performers = related_qualified_names(role, "rolePerformers")

    new_appointees = [e for e in appointees
                      if f"Person::{e['country_code']}::{e['employee_number']}" not in current_performers]

    role_properties = role["properties"]
    head_count_needed = len(current_performers) + len(new_appointees)
    team_id = role_qualified_name.split("::")[-1]
    if team_id in bucharest_team_ids and role_properties.get("headCount", 0) < head_count_needed:
        update_body = {
            "class": "UpdateElementRequestBody",
            "mergeUpdate": True,
            "properties": {"class": "PersonRoleProperties", "typeName": "PersonRole", "headCount": head_count_needed},
        }
        faiths_actor_client.update_actor_role(role_guid, update_body)

    for employee in new_appointees:
        faiths_actor_client.link_person_role_to_profile(role_guid, person_guids[employee["employee_number"]])

    print(f"{role_qualified_name}: {len(new_appointees)} appointed, {len(current_performers)} already appointed")

----

## Step 4: Appointing the system owners

The system inventory names a business owner for each subsystem of the Austin and Bucharest estates.  Gary turns each of these names into a role: an *asset-owner* *PersonRole* linked to the subsystem with the *AssignmentScope* relationship, and the named person appointed to it.  This means the owner of a system can be found by navigating from the system to its subsystem, and from there to the owner's profile and contact details.

----

In [ ]:
inventory_folder = "../../1. coco-data-hub/extending-the-systems-inventory/data/"
person_guids_by_name = {e["first_name"] + " " + e["surname"]: person_guids[e["employee_number"]] for e in employees}

for subsystems_file, site_name in [("coco_austin_subsystems.csv", "Austin"), ("ekg_subsystems.csv", "Bucharest")]:
    with open(inventory_folder + subsystems_file, newline="", encoding="utf-8") as csv_file:
        subsystems = list(csv.DictReader(csv_file))

    for subsystem in subsystems:
        subsystem_qualified_name = "ITSubsystem::" + subsystem["subsystem_id"] + "::" + subsystem["subsystem_name"]
        owner_name = subsystem["business_owner"].removeprefix("Dr. ")
        role_qualified_name = "AssetOwner::" + subsystem_qualified_name

        role_guid = get_guid(role_qualified_name)
        if role_guid is None:
            body = {
                "class": "NewElementRequestBody",
                "isOwnAnchor": True,
                "properties": {
                    "class": "PersonRoleProperties",
                    "typeName": "PersonRole",
                    "qualifiedName": role_qualified_name,
                    "identifier": subsystem["subsystem_id"] + "-OWNER",
                    "displayName": f"{site_name} {subsystem['subsystem_name']} Owner",
                    "description": f"Business owner of the {subsystem['subsystem_name']} subsystem at the {site_name} Site, accountable for the systems it contains.",
                    "actorRoleGroups": ["asset-owner"],
                    "headCountLimitSet": True,
                    "headCount": 1,
                },
            }
            role_guid = garys_actor_client.create_actor_role(body)
            scope_body = {
                "class": "NewRelationshipRequestBody",
                "properties": {"class": "AssignmentScopeProperties", "assignmentType": "Owner"},
            }
            garys_actor_client.link_assignment_scope(get_guid(subsystem_qualified_name), role_guid, scope_body)

        role = garys_actor_client.get_actor_role_by_guid(role_guid, output_format="JSON")
        if related_qualified_names(role, "rolePerformers"):
            print(f"{role_qualified_name} is already filled")
        else:
            garys_actor_client.link_person_role_to_profile(role_guid, person_guids_by_name[owner_name])
            print(f"{subsystem['business_owner']} appointed as owner of {subsystem_qualified_name}")

----

## Reviewing the results

Gheorghe Radu, the Finance Director at Bucharest, shows the result: his profile follows the same pattern as Gary's, with the new Bucharest work location (`12`) and the Romania country code (`RO`).  He is a member of the Bucharest Office Team and the owner of the ERP and Finance subsystem.

----

In [ ]:
profile = faiths_actor_client.get_actor_profile_by_guid(get_guid("Person::RO::EKG-005"), output_format="JSON")

profile_properties = profile["properties"]
print(f"{profile_properties['displayName']} - {profile_properties['jobTitle']}")
print(f"additionalProperties: {profile_properties['additionalProperties']}")
print()
for contact in profile.get("contactDetails", []):
    contact_properties = contact["relatedElement"]["properties"]
    print(f"{contact_properties['displayName']}: {contact_properties['contactMethodValue']}")
print()
for role in profile.get("performsRoles", []):
    print("Performs role: " + role["relatedElement"]["properties"]["qualifiedName"])

----

The teams at each site now list their named members.

----

In [ ]:
report = "| Team | Leader | Named members |\n|---|---|---|\n"
for team_id in ["9995", "0011", "0012", "0013", "9991", "0025", "0026", "0027"]:
    team = faiths_actor_client.get_actor_profile_by_guid(get_guid("Department::" + team_id), output_format="JSON")
    named = {"leader": [], "member": []}
    for kind, prefix in [("leader", "TeamLeader::"), ("member", "TeamMembers::")]:
        role = faiths_actor_client.get_actor_role_by_guid(get_guid(prefix + "Department::" + team_id), output_format="JSON")
        for performer in role.get("rolePerformers") or []:
            performer_properties = performer["relatedElement"]["properties"]
            if not performer_properties.get("qualifiedName", "").startswith("Person:Anon:"):
                named[kind].append(performer_properties.get("displayName"))
    report += f"| {team['properties']['displayName']} | {', '.join(named['leader'])} | {len(named['member'])} |\n"

display(Markdown(report))

----

## Next steps

The people at the acquired sites can now be appointed to governance roles, invited into communities and found as the owners of their systems.  The next notebook, [adding-procurement.ipynb](adding-procurement.ipynb), uses them to set up Procurement as a business area of its own.

----
License: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/),
Copyright Contributors to the ODPi Egeria project.